# XML Feature Extraction 

In [1]:
import os
import six
import pickle
import pandas as pd
import SimpleITK as sitk
from functools import reduce
import numpy as np
from scipy.stats import mode
from skimage.measure import find_contours
from skimage import io
import pydicom
import pylidc as pl
from pylidc.utils import consensus
import matplotlib.pyplot as plt

In [2]:
import sys
print(sys.executable)


C:\Users\user\anaconda3\envs\new_env\python.exe


In [ ]:
import os
import numpy as np
import pandas as pd
import pylidc as pl
from pylidc.utils import consensus

# Paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
path = "C:/Users/user/Desktop/PI_project/deeplung_features"
external_drive_path = "C:/Users/user/Desktop/PI_project"  # Path to save CSV

# ✅ Ensure output folder exists
os.makedirs(path, exist_ok=True)

# Initialize patient folders
patient_folders = sorted(os.listdir(input_directory))
nodule_id = 0

# Dictionary to collect nodule data
nodule_data = {
    'Nodule_id': [],
    'Patient_id': [],
    'Subtlety': [],
    'Internalstructure': [],
    'Calcification': [],
    'Sphericity': [],
    'Margin': [],
    'Lobulation': [],
    'Spiculation': [],
    'Texture': [],
    'Malignancy': []
}

# Process each patient
for patient_folder in patient_folders:
    patient_folder_path = os.path.join(input_directory, patient_folder)
    patient_id = os.path.basename(patient_folder_path)
    print(f"🔍 Processing patient: {patient_id}")
    
    # Query scans with annotations
    patient_scans = pl.query(pl.Scan).filter(pl.Scan.patient_id == patient_id, pl.Scan.annotations.any()).all()
    
    for scan in patient_scans:
        nods = scan.cluster_annotations()
        
        for anns in nods:
            # Get malignancy scores from all radiologists
            malignancy_scores = [ann.malignancy for ann in anns if ann.malignancy is not None]
            if len(malignancy_scores) == 0:
                continue
            
            avg_malignancy = np.mean(malignancy_scores)

            # Assign binary label like DeepLung
            if avg_malignancy >= 3.5:
                malignancy_label = "Malignant"
            elif avg_malignancy <= 2.5:
                malignancy_label = "Benign"
            else:
                continue  # Skip ambiguous nodules

            nodule_id += 1
            nodule_characteristics = []

            # Get 50% consensus mask
            cmask, cbbox, masks = consensus(anns, clevel=0.5, pad=[(20, 20), (20, 20), (0, 0)])

            for i, mask in enumerate(masks):
                characteristics = anns[i].feature_vals()
                nodule_characteristics.append(characteristics)

            if nodule_characteristics:
                # ✅ Save cmask
                filename = os.path.join(path, f"{nodule_id}_cmask.npy")
                np.save(filename, cmask)

                # Use first annotator’s features for simplicity
                feature_values = {
                    'Nodule_id': nodule_id,
                    'Patient_id': patient_id,
                    'Subtlety': nodule_characteristics[0][0],
                    'Internalstructure': nodule_characteristics[0][1],
                    'Calcification': nodule_characteristics[0][2],
                    'Sphericity': nodule_characteristics[0][3],
                    'Margin': nodule_characteristics[0][4],
                    'Lobulation': nodule_characteristics[0][5],
                    'Spiculation': nodule_characteristics[0][6],
                    'Texture': nodule_characteristics[0][7],
                    'Malignancy': malignancy_label
                }

                for key, value in feature_values.items():
                    nodule_data[key].append(value)

# Save features to CSV
features = pd.DataFrame(nodule_data)
csv_filename = os.path.join(external_drive_path, 'features_pylidc_deeplung_style.csv')
features.to_csv(csv_filename, index=False)

print("✅ DeepLung-style feature extraction complete and CSV saved.")


In [3]:
import os
import SimpleITK as sitk
import numpy as np
import pandas as pd
from radiomics import featureextractor

# Define paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params.yaml"
path = "C:/Users/user/Desktop/PI_project/std_limit_1.5"

# Initialize the radiomics feature extractor
extractor = featureextractor.RadiomicsFeatureExtractor(params_file)

# List to store features
dic_list = []

# Iterate through all the directories in the input directory
for nodule_id in os.listdir(input_directory):
    nodule_dir = os.path.join(input_directory, nodule_id)
    
    # Check if this is a directory and if it contains DICOM files
    if os.path.isdir(nodule_dir):
        dicom_files = [f for f in os.listdir(nodule_dir) if f.endswith('.dcm')]
        
        # If there are no DICOM files in this nodule, skip
        if not dicom_files:
            print(f"Skipping nodule {nodule_id}: No DICOM files found.")
            continue
        
        # Sort DICOM files if necessary (assuming the file names follow the correct order)
        dicom_files.sort()

        # Read DICOM files into a 3D SimpleITK image
        dicom_images = []
        for dicom_file in dicom_files:
            dicom_path = os.path.join(nodule_dir, dicom_file)
            dicom_image = sitk.ReadImage(dicom_path)
            dicom_images.append(dicom_image)
        
        # Stack the images into a 3D volume
        volume = sitk.JoinSeries(dicom_images)  # Join all slices into a 3D volume

        # Construct mask file (assuming you have masks like 'cmask.npy')
        mask_filename = f"{nodule_id}_cmask.npy"
        mask_path = os.path.join(path, mask_filename)
        
        # Check if mask exists
        if os.path.exists(mask_path):
            mask = np.load(mask_path)
            mask_sitk = sitk.GetImageFromArray(mask.astype(np.uint8))

            # Resample mask to match the resolution and size of the DICOM image
            resampler = sitk.ResampleImageFilter()
            resampler.SetSize(volume.GetSize())  # Set target size to match DICOM image
            resampler.SetOutputSpacing(volume.GetSpacing())  # Match spacing of DICOM image
            resampled_mask = resampler.Execute(mask_sitk)

            # Extract features using PyRadiomics
            try:
                features = extractor.execute(volume, resampled_mask, label=1)
                features['Nodule_id'] = int(nodule_id)
                dic_list.append(features)
            except Exception as e:
                print(f"Feature extraction failed for nodule {nodule_id}: {e}")
        else:
            print(f"Mask not found for nodule {nodule_id}. Skipping this nodule.")

# Convert to DataFrame
features_df = pd.DataFrame(dic_list)

# Save to CSV
csv_filename = os.path.join(path, 'features_radiomics_3d.csv')
features_df.to_csv(csv_filename, index=False)

# Done
print(f"\n✅ Features written to: {csv_filename}")
print(features_df.head())


Skipping nodule LIDC-IDRI-0001: No DICOM files found.
Skipping nodule LIDC-IDRI-0002: No DICOM files found.
Skipping nodule LIDC-IDRI-0003: No DICOM files found.
Skipping nodule LIDC-IDRI-0004: No DICOM files found.
Skipping nodule LIDC-IDRI-0005: No DICOM files found.
Skipping nodule LIDC-IDRI-0006: No DICOM files found.
Skipping nodule LIDC-IDRI-0007: No DICOM files found.
Skipping nodule LIDC-IDRI-0008: No DICOM files found.
Skipping nodule LIDC-IDRI-0009: No DICOM files found.
Skipping nodule LIDC-IDRI-0010: No DICOM files found.
Skipping nodule LIDC-IDRI-0011: No DICOM files found.
Skipping nodule LIDC-IDRI-0012: No DICOM files found.
Skipping nodule LIDC-IDRI-0013: No DICOM files found.
Skipping nodule LIDC-IDRI-0014: No DICOM files found.
Skipping nodule LIDC-IDRI-0015: No DICOM files found.
Skipping nodule LIDC-IDRI-0016: No DICOM files found.
Skipping nodule LIDC-IDRI-0017: No DICOM files found.
Skipping nodule LIDC-IDRI-0018: No DICOM files found.
Skipping nodule LIDC-IDRI-00